# Merchant Ranking System

This notebook develops an interpretable decision-support heuristic for BNPL merchant onboarding, not a statistically optimal prediction model.

It combines merchant value, customer reach/scale, growth, stability and market context into a business score, then applies a 10% fraud-safety adjustment using Member 4's consumer exposure and KNN-estimated merchant risk signals. KNN risk estimates are relative risk signals, not precise fraud probabilities.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

# Support execution from the repository root or the notebook directory.
cwd = Path.cwd().resolve()
PROJECT_ROOT = next(
    (path for path in (cwd, *cwd.parents)
     if (path / "member3_merchant_features").is_dir()
     and (path / "member4_fraud").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate the repository root.")
RESULTS_DIR = PROJECT_ROOT / "member5_ranking" / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

INPUT_PATH = (
    PROJECT_ROOT
    / "member3_merchant_features"
    / "results"
    / "merchant_features.parquet"
)

df = pd.read_parquet(INPUT_PATH)

print("Shape:", df.shape)
df.head()

Shape: (4422, 46)


,merchant_abn,merchant_name,merchant_category,merchant_pricing_level,merchant_take_rate_pct,has_merchant_master_record,total_transactions,total_revenue,avg_transaction_value,unique_consumers,...,census_single_parent_family_share,census_couple_with_children_family_share,census_unemployment_rate_published_pct,census_labour_force_participation_rate_published_pct,seifa_irsd_national_decile,seifa_irsad_national_decile,seifa_ier_national_decile,ato_taxable_income_or_loss_per_reporter,ato_salary_or_wages_per_recipient,ato_net_tax_payer_share
0,10023283211,Felis Limited,"furniture, home furnishings and equipment shop...",E,0.18,True,3261,703277.711451,215.663205,3032,...,0.137732,0.396714,4.545196,59.159350,5.426345,5.412987,5.455659,68260.371739,63998.111541,0.734878
1,10142254217,Arcu Ac Orci Corporation,"cable, satellite, and other pay television and...",B,4.22,True,3036,118356.146073,38.984238,2849,...,0.137717,0.397312,4.618711,59.135731,5.512892,5.524395,5.500595,69361.310505,64617.995761,0.734825
2,10165489824,Nunc Sed Company,"jewelry, watch, clock, and silverware shops",B,4.40,True,5,56180.473857,11236.094771,5,...,0.114423,0.490410,4.150000,63.625000,8.250000,8.000000,8.500000,76144.193842,63913.702304,0.782871
3,10187291046,Ultricies Dignissim Lacus Foundation,"watch, clock, and jewelry repair shops",B,3.29,True,336,39693.730387,118.136102,335,...,0.132108,0.393778,4.485274,59.633219,5.513793,5.434483,5.548276,68884.847450,65123.944932,0.734622
4,10192359162,Enim Condimentum PC,"music shops - musical instruments, pianos, and...",A,6.33,True,385,177980.505456,462.287027,383,...,0.136463,0.401961,4.413313,60.091331,5.744548,5.679128,5.732087,70416.453471,66239.804559,0.746924


## 1. Candidate Pool

The merchant feature table contains 4,422 merchants.

The eligible candidate pool contains 4,026 merchants with valid merchant master records.

Transaction-only orphan merchants are excluded because merchant category and take-rate information are unavailable, which prevents fair comparison on key BNPL value metrics.

In [2]:
eligible_df = df[df["has_merchant_master_record"]].copy()

print("All merchants:", len(df))
print("Eligible merchants:", len(eligible_df))
print("Excluded orphan merchants:", len(df) - len(eligible_df))

All merchants: 4422
Eligible merchants: 4026
Excluded orphan merchants: 396


## 2. Candidate Ranking Features

The preliminary ranking uses merchant-level features that capture business value, customer strength, growth, stability and market context.

CustomerScore uses `unique_consumers` to represent customer reach/scale. Repeat usage is inspected below but is not directly included in CustomerScore. KNN-based fraud safety is integrated after the business-only analysis.

In [3]:
candidate_features = [
    "estimated_bnpl_revenue",
    "total_revenue",
    "total_transactions",
    "unique_consumers",
    "avg_transaction_value",
    "repeat_consumer_share",
    "normalized_monthly_revenue_trend",
    "monthly_revenue_cv",
    "low_sample_growth_estimate",
    "regional_data_coverage_rate_all_sources_by_count",
    "census_median_household_income_weekly",
    "seifa_irsad_national_decile",
    "ato_taxable_income_or_loss_per_reporter",
]

eligible_df[candidate_features].describe().T

,count,mean,std,min,25%,50%,75%,max
estimated_bnpl_revenue,4026.0,23933.339418,5.849947e+04,12.313488,1470.210886,5146.420842,22933.624931,6.638703e+05
total_revenue,4026.0,535781.369254,1.218831e+06,10064.933919,36652.804201,141397.947807,628006.258580,9.857402e+06
total_transactions,4026.0,3381.687779,1.414670e+04,1.000000,93.000000,418.000000,2057.500000,2.895130e+05
unique_consumers,4026.0,2025.218579,3.875050e+03,1.000000,93.000000,415.500000,1961.000000,2.408100e+04
avg_transaction_value,4026.0,1159.595793,2.947518e+03,7.588294,118.099522,317.329914,836.237711,5.187664e+04
repeat_consumer_share,4026.0,0.051295,1.185795e-01,0.000000,0.000000,0.009718,0.041562,9.999169e-01
normalized_monthly_revenue_trend,4019.0,0.011004,4.624178e-02,-0.857143,0.006984,0.014355,0.020337,6.551908e-01
monthly_revenue_cv,4019.0,0.482603,4.134482e-01,0.133836,0.217327,0.321069,0.577596,3.872983e+00
regional_data_coverage_rate_all_sources_by_count,4026.0,0.806718,5.417902e-02,0.000000,0.796875,0.807871,0.819749,1.000000e+00
census_median_household_income_weekly,4026.0,1603.516538,8.298170e+01,806.500000,1587.309716,1604.879927,1620.820699,3.124000e+03


In [4]:
feature_missingness = (
    eligible_df[candidate_features]
    .isna()
    .mean()
    .sort_values(ascending=False)
    .to_frame("missing_rate")
)

feature_missingness

,missing_rate
normalized_monthly_revenue_trend,0.001739
monthly_revenue_cv,0.001739
ato_taxable_income_or_loss_per_reporter,0.000248
estimated_bnpl_revenue,0.000000
total_revenue,0.000000
total_transactions,0.000000
unique_consumers,0.000000
avg_transaction_value,0.000000
repeat_consumer_share,0.000000
low_sample_growth_estimate,0.000000


## 3. Feature Redundancy Check

Before assigning weights, pairwise correlations are examined to reduce double-counting of closely related merchant characteristics.

In [5]:
correlation_features = [
    "estimated_bnpl_revenue",
    "total_revenue",
    "total_transactions",
    "unique_consumers",
    "avg_transaction_value",
    "repeat_consumer_share",
    "normalized_monthly_revenue_trend",
    "monthly_revenue_cv",
    "census_median_household_income_weekly",
    "seifa_irsad_national_decile",
    "ato_taxable_income_or_loss_per_reporter",
]

correlation_matrix = eligible_df[correlation_features].corr().round(3)

correlation_matrix.to_csv(
    PROJECT_ROOT / "member5_ranking" / "results" / "feature_correlation_matrix.csv"
)

correlation_matrix

,estimated_bnpl_revenue,total_revenue,total_transactions,unique_consumers,avg_transaction_value,repeat_consumer_share,normalized_monthly_revenue_trend,monthly_revenue_cv,census_median_household_income_weekly,seifa_irsad_national_decile,ato_taxable_income_or_loss_per_reporter
estimated_bnpl_revenue,1.000,0.923,0.555,0.613,-0.026,0.636,0.031,-0.239,0.007,0.003,0.004
total_revenue,0.923,1.000,0.569,0.648,-0.022,0.663,0.032,-0.254,0.007,0.003,0.003
total_transactions,0.555,0.569,1.000,0.714,-0.081,0.820,0.020,-0.164,0.005,0.003,0.003
unique_consumers,0.613,0.648,0.714,1.000,-0.168,0.977,0.042,-0.347,0.011,0.006,0.006
avg_transaction_value,-0.026,-0.022,-0.081,-0.168,1.000,-0.142,-0.214,0.546,-0.026,-0.000,-0.085
repeat_consumer_share,0.636,0.663,0.820,0.977,-0.142,1.000,0.035,-0.292,0.010,0.005,0.005
normalized_monthly_revenue_trend,0.031,0.032,0.020,0.042,-0.214,0.035,1.000,-0.223,0.216,0.141,0.132
monthly_revenue_cv,-0.239,-0.254,-0.164,-0.347,0.546,-0.292,-0.223,1.000,-0.045,-0.024,-0.028
census_median_household_income_weekly,0.007,0.007,0.005,0.011,-0.026,0.010,0.216,-0.045,1.000,0.751,0.654
seifa_irsad_national_decile,0.003,0.003,0.003,0.006,-0.000,0.005,0.141,-0.024,0.751,1.000,0.519


## 4. Final Preliminary Feature Set

Highly correlated features were not simultaneously included in the preliminary score to reduce double-counting.

The selected non-fraud features are:

- **BNPL Value:** `estimated_bnpl_revenue`
- **Customer Reach/Scale:** `unique_consumers`
- **Growth:** `normalized_monthly_revenue_trend`
- **Stability:** `monthly_revenue_cv`
- **Market Context:** `seifa_irsad_national_decile`

The market score uses SEIFA IRSAD only; Census and ATO features remain contextual diagnostics. The subsequent fraud adjustment uses consumer exposure and KNN-estimated merchant risk.

## 5. Percentile Normalisation

To combine features measured on different scales, each selected metric is converted to a percentile score from 0 to 100.

Higher percentile scores indicate stronger merchant performance.

For metrics where lower values are preferred, such as revenue volatility, the percentile score is reversed.

In [6]:
score_df = eligible_df.copy()

# Higher is better
score_df["value_score"] = (
    score_df["estimated_bnpl_revenue"]
    .rank(pct=True, method="average") * 100
)

score_df["customer_score"] = (
    score_df["unique_consumers"]
    .rank(pct=True, method="average") * 100
)

score_df["growth_score"] = (
    score_df["normalized_monthly_revenue_trend"]
    .rank(pct=True, method="average") * 100
)

score_df["market_score"] = (
    score_df["seifa_irsad_national_decile"]
    .rank(pct=True, method="average") * 100
)

# Lower volatility is better
score_df["stability_score"] = (
    100
    - score_df["monthly_revenue_cv"]
    .rank(pct=True, method="average") * 100
)

score_df[
    [
        "value_score",
        "customer_score",
        "growth_score",
        "stability_score",
        "market_score",
    ]
].describe().round(2)

,value_score,customer_score,growth_score,stability_score,market_score
count,4026.00,4026.00,4019.00,4019.00,4026.00
mean,50.01,50.01,50.01,49.99,50.01
std,28.87,28.87,28.87,28.87,28.87
min,0.02,0.10,0.02,0.00,0.02
25%,25.02,25.06,25.02,24.99,25.02
50%,50.01,50.02,50.01,49.99,50.01
75%,75.01,75.00,75.01,74.98,75.01
max,100.00,99.98,100.00,99.98,99.98


## 6. Preliminary Weighted Score

A preliminary balanced weighting scheme is used before fraud risk is integrated.

The current weights prioritise direct BNPL value and customer scale, while still allowing growth, stability and market context to influence the ranking.

In [7]:
# Neutral score for merchants whose growth/stability cannot be estimated
score_df["growth_score"] = score_df["growth_score"].fillna(50)
score_df["stability_score"] = score_df["stability_score"].fillna(50)

In [8]:
PRELIMINARY_WEIGHTS = {
    "value_score": 0.30,
    "customer_score": 0.25,
    "growth_score": 0.20,
    "stability_score": 0.15,
    "market_score": 0.10,
}

score_df["preliminary_score"] = sum(
    weight * score_df[column]
    for column, weight in PRELIMINARY_WEIGHTS.items()
)

score_df["preliminary_rank"] = (
    score_df["preliminary_score"]
    .rank(method="min", ascending=False)
    .astype(int)
)

preliminary_top_20 = (
    score_df
    .sort_values("preliminary_score", ascending=False)
    [
        [
            "preliminary_rank",
            "merchant_abn",
            "merchant_name",
            "merchant_category",
            "preliminary_score",
            "value_score",
            "customer_score",
            "growth_score",
            "stability_score",
            "market_score",
        ]
    ]
    .head(20)
)

preliminary_top_20

,preliminary_rank,merchant_abn,merchant_name,merchant_category,preliminary_score,value_score,customer_score,growth_score,stability_score,market_score
3966,1,90568944804,Diam Eu Dolor LLC,tent and awning shops,87.168863,99.230005,93.169399,75.167952,82.483205,67.014406
1414,2,38090089066,Interdum Feugiat Sed Inc.,"furniture, home furnishings and equipment shop...",86.800764,98.683557,99.254844,61.607365,93.928838,59.711873
869,3,27326652377,Tellus Aenean Corporation,"music shops - musical instruments, pianos, and...",86.253316,99.403875,89.195231,75.267479,86.165713,61.549925
3782,4,86772484982,Posuere Cubilia Curae LLC,"digital goods: books, movies, music",85.785250,95.032290,95.355191,59.442647,98.283155,68.057625
3692,5,84703983173,Amet Consulting,"computer programming , data processing, and in...",85.765124,96.572280,99.006458,68.847972,89.524757,48.435171
4259,6,96680767841,Ornare Limited,motor vehicle supplies and new parts,85.723766,99.850969,98.435171,63.772083,90.097039,48.907104
3772,7,86578477987,Leo In Consulting,"watch, clock, and jewelry repair shops",85.650047,99.925484,99.975161,55.113212,95.172929,53.800298
1956,8,49212265466,Auctor Company,"florists supplies, nursery stock, and flowers",85.552242,99.056135,98.782911,60.014929,94.575765,49.503229
2367,9,57699602880,Tellus Id Institute,"stationery, office supplies and printing and w...",85.490283,97.441629,82.103825,66.857427,97.885046,76.775956
1917,10,48534649627,Dignissim Maecenas Foundation,"opticians, optical goods, and eyeglasses",85.327189,99.975161,99.428713,59.119184,89.997512,51.539990


In [9]:
preliminary_top_20.to_csv(
    PROJECT_ROOT / "member5_ranking" / "results" / "preliminary_top_20.csv",
    index=False
)

For merchants whose growth or stability could not be estimated because of insufficient activity in the fixed analysis window, a neutral percentile score of 50 is assigned. These merchants remain flagged as low-confidence rather than being rewarded or penalised for unavailable estimates.

## 7. Sensitivity Analysis

To test whether the merchant ranking is overly dependent on one specific weighting choice, three reasonable weighting scenarios are compared.

The scenarios represent:
- a balanced business view,
- a stronger emphasis on merchant value,
- a stronger emphasis on future growth.

Ranking robustness is assessed using Top 100 overlap and merchant rank stability.

In [10]:
SCENARIOS = {
    "balanced": {
        "value_score": 0.30,
        "customer_score": 0.25,
        "growth_score": 0.20,
        "stability_score": 0.15,
        "market_score": 0.10,
    },
    "value_focused": {
        "value_score": 0.40,
        "customer_score": 0.25,
        "growth_score": 0.15,
        "stability_score": 0.15,
        "market_score": 0.05,
    },
    "growth_focused": {
        "value_score": 0.25,
        "customer_score": 0.20,
        "growth_score": 0.30,
        "stability_score": 0.15,
        "market_score": 0.10,
    },
}

for scenario_name, weights in SCENARIOS.items():
    score_col = f"{scenario_name}_score"
    rank_col = f"{scenario_name}_rank"

    score_df[score_col] = sum(
        weight * score_df[column]
        for column, weight in weights.items()
    )

    score_df[rank_col] = (
        score_df[score_col]
        .rank(method="min", ascending=False)
        .astype(int)
    )

In [11]:
top100_sets = {}

for scenario_name in SCENARIOS:
    score_col = f"{scenario_name}_score"

    top100_sets[scenario_name] = set(
        score_df
        .nlargest(100, score_col)["merchant_abn"]
    )

overlap_results = []

scenario_names = list(SCENARIOS.keys())

for i in range(len(scenario_names)):
    for j in range(i + 1, len(scenario_names)):

        scenario_a = scenario_names[i]
        scenario_b = scenario_names[j]

        overlap_count = len(
            top100_sets[scenario_a]
            & top100_sets[scenario_b]
        )

        overlap_results.append({
            "scenario_a": scenario_a,
            "scenario_b": scenario_b,
            "top100_overlap_count": overlap_count,
            "top100_overlap_rate": overlap_count / 100,
        })

overlap_df = pd.DataFrame(overlap_results)

overlap_df

,scenario_a,scenario_b,top100_overlap_count,top100_overlap_rate
0,balanced,value_focused,85,0.85
1,balanced,growth_focused,80,0.80
2,value_focused,growth_focused,66,0.66


In [12]:
rank_columns = [
    "balanced_rank",
    "value_focused_rank",
    "growth_focused_rank",
]

score_df["best_rank"] = score_df[rank_columns].min(axis=1)
score_df["worst_rank"] = score_df[rank_columns].max(axis=1)

score_df["rank_range"] = (
    score_df["worst_rank"]
    - score_df["best_rank"]
)

rank_stability = (
    score_df[
        [
            "merchant_abn",
            "merchant_name",
            "balanced_rank",
            "value_focused_rank",
            "growth_focused_rank",
            "rank_range",
        ]
    ]
    .sort_values("rank_range")
)

rank_stability.head(20)

,merchant_abn,merchant_name,balanced_rank,value_focused_rank,growth_focused_rank,rank_range
4236,96190048310,Penatibus Et Inc.,4023,4023,4023,0
178,13747603419,Fermentum Vel Mauris Institute,4024,4024,4024,0
3034,71475747855,Aliquet Molestie Corporation,4025,4025,4025,0
985,29623808496,Iaculis Odio Nam Foundation,4026,4026,4026,0
4144,93915598279,Molestie Pharetra Nibh LLP,4022,4021,4022,1
373,17507773571,Libero Nec Limited,4020,4019,4020,1
2745,65959377833,Lobortis Augue Ltd,4021,4022,4021,1
3608,82882460979,Sem Molestie Foundation,4011,4010,4011,1
280,15704713883,Faucibus Ut Nulla Ltd,4016,4016,4017,1
3994,91062920626,Ac Inc.,4019,4018,4019,1


In [13]:
top100_stability = (
    score_df[score_df["balanced_rank"] <= 100]
    [
        [
            "merchant_abn",
            "merchant_name",
            "balanced_rank",
            "value_focused_rank",
            "growth_focused_rank",
            "rank_range",
        ]
    ]
    .sort_values("balanced_rank")
)

top100_stability

,merchant_abn,merchant_name,balanced_rank,value_focused_rank,growth_focused_rank,rank_range
3966,90568944804,Diam Eu Dolor LLC,1,4,1,3
1414,38090089066,Interdum Feugiat Sed Inc.,2,1,4,3
869,27326652377,Tellus Aenean Corporation,3,13,2,11
3782,86772484982,Posuere Cubilia Curae LLC,4,28,8,24
3692,84703983173,Amet Consulting,5,8,5,3
...,...,...,...,...,...,...
989,29639699851,Sodales Elit Erat Corporation,96,87,135,48
2696,64732735902,Imperdiet Non Vestibulum Institute,97,138,82,56
3348,77590625261,Sed Diam Foundation,98,76,189,113
1300,35556933338,Semper Cursus Integer Limited,99,90,130,40


In [14]:
top100_stability["rank_range"].describe()

count    100.000000
mean      49.510000
std       32.408533
min        3.000000
25%       25.500000
50%       44.000000
75%       70.250000
max      179.000000
Name: rank_range, dtype: float64

In [15]:
top100_most_sensitive = (
    top100_stability
    .sort_values("rank_range", ascending=False)
    .head(20)
)

top100_most_sensitive

,merchant_abn,merchant_name,balanced_rank,value_focused_rank,growth_focused_rank,rank_range
2226,55069630872,A Nunc Corp.,93,200,21,179
4119,93558142492,Dolor Quisque Inc.,94,56,183,127
1969,49505931725,Suspendisse Ac Associates,71,42,164,122
3348,77590625261,Sed Diam Foundation,98,76,189,113
3085,72472909171,Nullam Consulting,73,43,154,111
4175,94729574738,Scelerisque Corporation,72,115,10,105
2396,58454491168,Diam At Foundation,62,40,140,100
2471,60111071436,Imperdiet Non LLC,81,141,45,96
3479,80324045558,Ipsum Dolor Sit Corporation,84,59,155,96
1128,32361057556,Orci In Consequat Corporation,58,31,125,94


In [16]:
top20_stability = (
    score_df[score_df["balanced_rank"] <= 20]
    [
        [
            "merchant_abn",
            "merchant_name",
            "balanced_rank",
            "value_focused_rank",
            "growth_focused_rank",
            "rank_range",
        ]
    ]
    .sort_values("balanced_rank")
)

top20_stability

,merchant_abn,merchant_name,balanced_rank,value_focused_rank,growth_focused_rank,rank_range
3966,90568944804,Diam Eu Dolor LLC,1,4,1,3
1414,38090089066,Interdum Feugiat Sed Inc.,2,1,4,3
869,27326652377,Tellus Aenean Corporation,3,13,2,11
3782,86772484982,Posuere Cubilia Curae LLC,4,28,8,24
3692,84703983173,Amet Consulting,5,8,5,3
4259,96680767841,Ornare Limited,6,3,9,6
3772,86578477987,Leo In Consulting,7,2,32,30
1956,49212265466,Auctor Company,8,5,14,9
2367,57699602880,Tellus Id Institute,9,52,3,49
1917,48534649627,Dignissim Maecenas Foundation,10,6,25,19


In [17]:
# Number of balanced Top 100 merchants that remain Top 100 in all scenarios
stable_top100_count = (
    (score_df["balanced_rank"] <= 100)
    & (score_df["value_focused_rank"] <= 100)
    & (score_df["growth_focused_rank"] <= 100)
).sum()

print("Balanced Top 100 retained in all scenarios:", stable_top100_count)

Balanced Top 100 retained in all scenarios: 66


In [18]:
# Summary for the balanced Top 20
top20_stability["rank_range"].describe()

count    20.000000
mean     18.250000
std      13.466821
min       3.000000
25%       6.750000
50%      17.000000
75%      26.000000
max      49.000000
Name: rank_range, dtype: float64

### Sensitivity Analysis Interpretation

The balanced ranking is moderately robust to reasonable changes in weighting assumptions.

The Top 100 overlap is 85% between the balanced and value-focused scenarios and 80% between the balanced and growth-focused scenarios. The overlap between the two more extreme scenarios falls to 66%, indicating that merchant selection changes when current commercial value and future growth are prioritised differently.

Among the balanced Top 100, 66 merchants remain in the Top 100 under all three scenarios. These merchants can therefore be treated as a relatively robust core recommendation set.

Exact rank positions are more sensitive than Top 100 membership. The median rank range among the balanced Top 100 is 44 positions, while the median rank range among the balanced Top 20 is only 17 positions.

This indicates that the highest-ranked merchants are generally more stable, while sensitivity is concentrated among merchants closer to the Top 100 selection boundary.

## 8. Fraud Risk Integration

Member 4's `member4_fraud/result/knn_fraud_risk_scores.csv` already combines consumer fraud exposure and KNN-estimated merchant fraud risk.

The balanced baseline is:

`fraud_risk_index = 0.50 * consumer_exposure_percentile + 0.50 * knn_merchant_risk_percentile`

`risk_safety_score = 100 * (1 - fraud_risk_index)`

The 50/50 composition is a balanced baseline, not a statistically optimal weight. These are relative fraud-risk signals, not precise fraud probabilities. The formula applies where both components are available. Seven eligible merchants lack consumer exposure; Member 4 already supplies KNN-only combined scores for these rows. Member 5 consumes the supplied scores unchanged and performs no missing-fraud filling. Missing combined scores or reliability diagnostics stop execution.

In [19]:
FRAUD_PATH = (
    PROJECT_ROOT
    / "member4_fraud"
    / "result"
    / "knn_fraud_risk_scores.csv"
)

fraud_df = pd.read_csv(FRAUD_PATH, dtype={"merchant_abn": str})

print("Project root:", PROJECT_ROOT)
print("Fraud profile shape:", fraud_df.shape)
print("Unique merchants:", fraud_df["merchant_abn"].nunique())

fraud_df.head()

Project root: /Users/xiexinyu/Desktop/MAST30034-Project-2-Group-50
Fraud profile shape: (4422, 19)
Unique merchants: 4422


,merchant_abn,merchant_name,consumer_exposure_percentile,knn_score,knn_merchant_risk_percentile,knn_mean_neighbor_distance,knn_max_neighbor_distance,knn_distance_percentile,knn_confidence_score,knn_out_of_distribution,knn_score_source,fraud_risk_70c_30knn,risk_safety_70c_30knn,fraud_risk_50c_50knn,risk_safety_50c_50knn,fraud_risk_30c_70knn,risk_safety_30c_70knn,fraud_risk_index,risk_safety_score
0,10023283211,Felis Limited,0.579878,0.509525,0.334314,4.486615,5.220752,0.967213,0.032787,True,full_model_for_unlabelled_merchant,0.506208,49.379160,0.457096,54.290443,0.407983,59.201726,0.457096,54.290443
1,10142254217,Arcu Ac Orci Corporation,0.447315,0.657310,0.692830,3.447449,4.239421,0.967213,0.032787,True,full_model_for_unlabelled_merchant,0.520969,47.903077,0.570072,42.992779,0.619175,38.082480,0.570072,42.992779
2,10165489824,Nunc Sed Company,0.000000,0.313441,0.111061,3.130178,4.717325,0.934426,0.065574,False,full_model_for_unlabelled_merchant,0.033318,96.668175,0.055530,94.446958,0.077743,92.225741,0.055530,94.446958
3,10187291046,Ultricies Dignissim Lacus Foundation,0.114888,0.658932,0.765890,1.129005,1.470959,0.426230,0.573770,False,full_model_for_unlabelled_merchant,0.310189,68.981150,0.440389,55.961105,0.570589,42.941060,0.440389,55.961105
4,10192359162,Enim Condimentum PC,0.902334,0.652009,0.566614,1.489107,2.008204,0.622951,0.377049,False,full_model_for_unlabelled_merchant,0.801618,19.838202,0.734474,26.552605,0.667330,33.267007,0.734474,26.552605


In [20]:
fraud_columns = [
    "merchant_abn",
    "fraud_risk_index",
    "risk_safety_score",

    "fraud_risk_70c_30knn",
    "risk_safety_70c_30knn",

    "fraud_risk_50c_50knn",
    "risk_safety_50c_50knn",

    "fraud_risk_30c_70knn",
    "risk_safety_30c_70knn",

    "consumer_exposure_percentile",
    "knn_merchant_risk_percentile",
    "knn_confidence_score",
    "knn_out_of_distribution",
]

fraud_for_ranking = fraud_df[fraud_columns].copy()

required_fraud_columns = [column for column in fraud_columns if column != "consumer_exposure_percentile"]
if fraud_for_ranking[required_fraud_columns].isna().any().any():
    raise ValueError("Required KNN fraud inputs contain missing values.")
if fraud_for_ranking["merchant_abn"].duplicated().any():
    raise ValueError("KNN fraud inputs contain duplicate merchant identifiers.")
for suffix, consumer_weight in [("70c_30knn", 0.70), ("50c_50knn", 0.50), ("30c_70knn", 0.30)]:
    expected_risk = (
        consumer_weight * fraud_for_ranking["consumer_exposure_percentile"]
        + (1 - consumer_weight) * fraud_for_ranking["knn_merchant_risk_percentile"]
    )
    both_available = expected_risk.notna()
    np.testing.assert_allclose(
        fraud_for_ranking.loc[both_available, f"fraud_risk_{suffix}"],
        expected_risk.loc[both_available],
    )
    np.testing.assert_allclose(
        fraud_for_ranking[f"risk_safety_{suffix}"], 100 * (1 - fraud_for_ranking[f"fraud_risk_{suffix}"])
    )
np.testing.assert_allclose(fraud_for_ranking["fraud_risk_index"], fraud_for_ranking["fraud_risk_50c_50knn"])
np.testing.assert_allclose(fraud_for_ranking["risk_safety_score"], fraud_for_ranking["risk_safety_50c_50knn"])

fraud_for_ranking.to_csv(
    PROJECT_ROOT
    / "member5_ranking"
    / "results"
    / "fraud_for_ranking.csv",
    index=False
)

print("Shape:", fraud_for_ranking.shape)
fraud_for_ranking.head()

Shape: (4422, 13)


,merchant_abn,fraud_risk_index,risk_safety_score,fraud_risk_70c_30knn,risk_safety_70c_30knn,fraud_risk_50c_50knn,risk_safety_50c_50knn,fraud_risk_30c_70knn,risk_safety_30c_70knn,consumer_exposure_percentile,knn_merchant_risk_percentile,knn_confidence_score,knn_out_of_distribution
0,10023283211,0.457096,54.290443,0.506208,49.379160,0.457096,54.290443,0.407983,59.201726,0.579878,0.334314,0.032787,True
1,10142254217,0.570072,42.992779,0.520969,47.903077,0.570072,42.992779,0.619175,38.082480,0.447315,0.692830,0.032787,True
2,10165489824,0.055530,94.446958,0.033318,96.668175,0.055530,94.446958,0.077743,92.225741,0.000000,0.111061,0.065574,False
3,10187291046,0.440389,55.961105,0.310189,68.981150,0.440389,55.961105,0.570589,42.941060,0.114888,0.765890,0.573770,False
4,10192359162,0.734474,26.552605,0.801618,19.838202,0.734474,26.552605,0.667330,33.267007,0.902334,0.566614,0.377049,False


In [21]:
score_df["merchant_abn"] = score_df["merchant_abn"].astype(str)
fraud_for_ranking["merchant_abn"] = fraud_for_ranking["merchant_abn"].astype(str)

final_df = score_df.merge(
    fraud_for_ranking,
    on="merchant_abn",
    how="left",
    validate="one_to_one",
)

print("Before merge:", len(score_df))
print("After merge:", len(final_df))

print(
    "Missing fraud risk:",
    final_df["fraud_risk_index"].isna().sum()
)

if final_df[required_fraud_columns].isna().any().any():
    raise ValueError("Eligible merchants are missing required KNN fraud inputs.")
if final_df["balanced_score"].isna().any():
    raise ValueError("Eligible merchants are missing business scores.")

print("Eligible merchants without consumer exposure (upstream scores retained):",
      final_df["consumer_exposure_percentile"].isna().sum())

Before merge: 4026
After merge: 4026
Missing fraud risk: 0
Eligible merchants without consumer exposure (upstream scores retained): 7


In [22]:
final_df["risk_safety_score"].describe()

count    4026.000000
mean       49.143295
std        20.731964
min         1.382075
25%        34.189044
50%        48.940766
75%        65.137231
max        99.762374
Name: risk_safety_score, dtype: float64

## 9. Final Ranking with Fraud Risk

The final merchant ranking combines business performance with fraud-risk safety.

The business component uses the balanced business score developed earlier. Fraud risk is converted into a safety score, where higher values indicate lower relative fraud risk.

The baseline final score assigns 90% weight to business performance and 10% weight to fraud-risk safety.

In [23]:
final_df["final_score"] = (
    0.90 * final_df["balanced_score"]
    + 0.10 * final_df["risk_safety_score"]
)

final_df["final_rank"] = (
    final_df["final_score"]
    .rank(method="min", ascending=False)
    .astype(int)
)


final_df[
    [
        "merchant_abn",
        "merchant_name",
        "balanced_score",
        "risk_safety_score",
        "final_score",
        "balanced_rank",
        "final_rank",
    ]
].sort_values("final_rank").head()

,merchant_abn,merchant_name,balanced_score,risk_safety_score,final_score,balanced_rank,final_rank
1301,38090089066,Interdum Feugiat Sed Inc.,86.800764,69.637978,85.084485,2,1
3614,90568944804,Diam Eu Dolor LLC,87.168863,63.826405,84.834617,1,2
3369,84703983173,Amet Consulting,85.765124,74.419880,84.630599,5,3
3443,86578477987,Leo In Consulting,85.650047,74.543876,84.539430,7,4
803,27326652377,Tellus Aenean Corporation,86.253316,67.360254,84.364009,3,5


In [24]:
final_df["rank_change_after_fraud"] = (
    final_df["balanced_rank"] - final_df["final_rank"]
)

final_df["rank_change_after_fraud"].describe()

count    4026.000000
mean        0.000000
std       113.404551
min      -299.000000
25%       -71.750000
50%        -3.000000
75%        63.000000
max       415.000000
Name: rank_change_after_fraud, dtype: float64

### Fraud-composition sensitivity

Hold the final fraud-safety weight at 10% and compare 70% consumer / 30% KNN, 50% / 50%, and 30% / 70%. The 50/50 composition is a balanced baseline, not a statistically optimal mix.


In [25]:
final_df["final_score_70c_30knn"] = (
    0.90 * final_df["balanced_score"]
    + 0.10 * final_df["risk_safety_70c_30knn"]
)

final_df["final_score_50c_50knn"] = (
    0.90 * final_df["balanced_score"]
    + 0.10 * final_df["risk_safety_50c_50knn"]
)

final_df["final_score_30c_70knn"] = (
    0.90 * final_df["balanced_score"]
    + 0.10 * final_df["risk_safety_30c_70knn"]
)

In [26]:
np.testing.assert_allclose(final_df["final_score_50c_50knn"], final_df["final_score"])

In [27]:
top100_70 = set(
    final_df.nlargest(100, "final_score_70c_30knn")["merchant_abn"]
)

top100_50 = set(
    final_df.nlargest(100, "final_score_50c_50knn")["merchant_abn"]
)

top100_30 = set(
    final_df.nlargest(100, "final_score_30c_70knn")["merchant_abn"]
)


rank_70 = final_df["final_score_70c_30knn"].rank()
rank_50 = final_df["final_score_50c_50knn"].rank()
rank_30 = final_df["final_score_30c_70knn"].rank()


sensitivity_summary = pd.DataFrame({
    "comparison": [
        "70/30 vs 50/50",
        "30/70 vs 50/50",
        "70/30 vs 30/70",
    ],
    "top100_overlap": [
        len(top100_70 & top100_50),
        len(top100_30 & top100_50),
        len(top100_70 & top100_30),
    ],
    "spearman_rank_correlation": [
        rank_70.corr(rank_50),
        rank_30.corr(rank_50),
        rank_70.corr(rank_30),
    ],
})

sensitivity_summary

,comparison,top100_overlap,spearman_rank_correlation
0,70/30 vs 50/50,96,0.999373
1,30/70 vs 50/50,95,0.999370
2,70/30 vs 30/70,91,0.997521


In [28]:
sensitivity_summary.to_csv(
    PROJECT_ROOT
    / "member5_ranking"
    / "results"
    / "fraud_composition_final_ranking_sensitivity.csv",
    index=False
)

Top 100 overlaps are 96 (70/30 vs 50/50), 95 (30/70 vs 50/50), and 91 (70/30 vs 30/70). Corresponding Spearman correlations are approximately 0.99937, 0.99937 and 0.99752. Overall ordering is very similar, with some selection changes near the Top 100 boundary.

### KNN reliability diagnostics


In [29]:
top100_final = (
    final_df
    .sort_values("final_rank")
    .head(100)
    .copy()
)

print(
    "Top 100 KNN out-of-distribution count:",
    top100_final["knn_out_of_distribution"].sum()
)

print(
    "Top 100 KNN out-of-distribution share:",
    top100_final["knn_out_of_distribution"].mean()
)

print(
    "Top 100 mean KNN confidence:",
    top100_final["knn_confidence_score"].mean()
)

print(
    "All eligible mean KNN confidence:",
    final_df["knn_confidence_score"].mean()
)

Top 100 KNN out-of-distribution count: 12
Top 100 KNN out-of-distribution share: 0.12
Top 100 mean KNN confidence: 0.44196721311475407
All eligible mean KNN confidence: 0.19361852874349514


In [30]:
top100_final[
    [
        "merchant_abn",
        "merchant_name",
        "final_rank",
        "final_score",
        "knn_confidence_score",
        "knn_out_of_distribution",
    ]
].head(20)

,merchant_abn,merchant_name,final_rank,final_score,knn_confidence_score,knn_out_of_distribution
1301,38090089066,Interdum Feugiat Sed Inc.,1,85.084485,0.918033,False
3614,90568944804,Diam Eu Dolor LLC,2,84.834617,0.327869,False
3369,84703983173,Amet Consulting,3,84.630599,0.459016,False
3443,86578477987,Leo In Consulting,4,84.539430,0.639344,False
803,27326652377,Tellus Aenean Corporation,5,84.364009,0.245902,False
1401,40515428545,Elit Sed Consequat Associates,6,84.276448,0.557377,False
348,17488304283,Posuere Cubilia Curae Corporation,7,84.273831,0.377049,False
1798,49212265466,Auctor Company,8,84.246497,0.885246,False
3453,86772484982,Posuere Cubilia Curae LLC,9,84.237758,0.065574,False
3881,96680767841,Ornare Limited,10,84.216013,0.967213,False


### Fraud-model reliability in the final recommendations

The KNN fraud model also provides a confidence diagnostic based on merchant similarity to the labelled training sample.

Among the final Top 100 merchants:

- 12 merchants (12%) are flagged as out-of-distribution;
- the mean KNN confidence score is 0.442, compared with 0.194 across all eligible merchants;
- none of the Top 20 merchants are flagged as out-of-distribution.

This suggests that the highest-ranked recommendations generally have stronger support from the KNN similarity structure than the wider merchant population. However, fraud-risk estimates for the 12 out-of-distribution merchants should be interpreted with greater caution rather than treated as precise fraud probabilities.

In [31]:
reliability_summary = pd.DataFrame({
    "metric": [
        "top100_ood_count",
        "top100_ood_share",
        "top100_mean_knn_confidence",
        "all_eligible_mean_knn_confidence",
        "top20_ood_count",
    ],
    "value": [
        int(top100_final["knn_out_of_distribution"].sum()),
        top100_final["knn_out_of_distribution"].mean(),
        top100_final["knn_confidence_score"].mean(),
        final_df["knn_confidence_score"].mean(),
        int(top100_final.head(20)["knn_out_of_distribution"].sum()),
    ],
})

reliability_summary.to_csv(
    PROJECT_ROOT
    / "member5_ranking"
    / "results"
    / "top100_knn_reliability_summary.csv",
    index=False
)

reliability_summary

,metric,value
0,top100_ood_count,12.000000
1,top100_ood_share,0.120000
2,top100_mean_knn_confidence,0.441967
3,all_eligible_mean_knn_confidence,0.193619
4,top20_ood_count,0.000000


In [32]:
largest_rank_changes = (
    final_df[
        [
            "merchant_abn",
            "merchant_name",
            "balanced_rank",
            "final_rank",
            "rank_change_after_fraud",
            "balanced_score",
            "risk_safety_score",
            "knn_confidence_score",
            "knn_out_of_distribution",
        ]
    ]
    .assign(
        absolute_rank_change=lambda x:
        x["rank_change_after_fraud"].abs()
    )
    .sort_values("absolute_rank_change", ascending=False)
    .head(20)
)

largest_rank_changes

,merchant_abn,merchant_name,balanced_rank,final_rank,rank_change_after_fraud,balanced_score,risk_safety_score,knn_confidence_score,knn_out_of_distribution,absolute_rank_change
3371,84787662573,Varius Orci Inc.,2486,2071,415,42.920730,99.286794,0.032787,True,415
3123,79081516258,A Neque LLP,2633,2224,409,40.513710,99.570130,0.032787,True,409
1338,38918664617,Faucibus Foundation,2466,2062,404,43.238587,97.158508,0.032787,True,404
3097,78577106740,Donec Consulting,2464,2061,403,43.243667,97.213129,0.032787,True,403
102,12240378188,Ut Ipsum Corp.,2636,2236,400,40.473307,98.471720,0.032787,True,400
3549,88745863985,Fringilla Ornare Placerat Institute,2978,2588,390,35.101828,98.596495,0.032787,True,390
1931,52448445000,Feugiat Placerat Industries,2631,2246,385,40.518563,96.818214,0.032787,True,385
573,22503967537,Iaculis Quis LLC,3088,2711,377,33.074534,99.230472,0.032787,True,377
3570,89133730546,Adipiscing Fringilla Porttitor Ltd,3100,2725,375,32.900993,98.664230,0.032787,True,375
990,31416331470,Quis Turpis Corporation,3037,2663,374,33.906411,98.653372,0.032787,True,374


In [33]:
business_top100 = set(
    final_df.loc[final_df["balanced_rank"] <= 100, "merchant_abn"]
)

final_top100 = set(
    final_df.loc[final_df["final_rank"] <= 100, "merchant_abn"]
)

top100_overlap = len(business_top100 & final_top100)

print("Business vs final Top 100 overlap:", top100_overlap)
print("Overlap rate:", top100_overlap / 100)

Business vs final Top 100 overlap: 90
Overlap rate: 0.9


### Initial Fraud Adjustment Check

The overlap above measures how the 10% fraud-safety adjustment changes business-only Top 100 membership. The rank-change table retains KNN confidence and out-of-distribution diagnostics to help interpret relative risk signals.

Business performance retains 90% of the final score. Fraud safety is a baseline adjustment; the sensitivity analyses assess how selection changes under alternative risk assumptions.

## 10. Sensitivity to Fraud Weight

The baseline final ranking assigns 10% weight to fraud-risk safety.

To test whether the final merchant selection is overly dependent on this assumption, alternative fraud weights of 5% and 15% are compared with the 10% baseline.

In [34]:
FRAUD_WEIGHTS = {
    "fraud_5": 0.05,
    "fraud_10": 0.10,
    "fraud_15": 0.15,
}

for name, fraud_weight in FRAUD_WEIGHTS.items():
    business_weight = 1 - fraud_weight

    score_col = f"{name}_score"
    rank_col = f"{name}_rank"

    final_df[score_col] = (
        business_weight * final_df["balanced_score"]
        + fraud_weight * final_df["risk_safety_score"]
    )

    final_df[rank_col] = (
        final_df[score_col]
        .rank(method="min", ascending=False)
        .astype(int)
    )

In [35]:
fraud_top100_sets = {}

for name in FRAUD_WEIGHTS:
    rank_col = f"{name}_rank"

    fraud_top100_sets[name] = set(
        final_df.loc[
            final_df[rank_col] <= 100,
            "merchant_abn"
        ]
    )

fraud_overlap_results = []

scenario_names = list(FRAUD_WEIGHTS.keys())

for i in range(len(scenario_names)):
    for j in range(i + 1, len(scenario_names)):
        a = scenario_names[i]
        b = scenario_names[j]

        overlap = len(
            fraud_top100_sets[a]
            & fraud_top100_sets[b]
        )

        fraud_overlap_results.append({
            "scenario_a": a,
            "scenario_b": b,
            "top100_overlap_count": overlap,
            "top100_overlap_rate": overlap / 100,
        })

fraud_weight_overlap_df = pd.DataFrame(
    fraud_overlap_results
)

fraud_weight_overlap_df

,scenario_a,scenario_b,top100_overlap_count,top100_overlap_rate
0,fraud_5,fraud_10,94,0.94
1,fraud_5,fraud_15,86,0.86
2,fraud_10,fraud_15,92,0.92


In [36]:
fraud_rank_cols = [
    "fraud_5_rank",
    "fraud_10_rank",
    "fraud_15_rank",
]

final_df["fraud_weight_best_rank"] = (
    final_df[fraud_rank_cols].min(axis=1)
)

final_df["fraud_weight_worst_rank"] = (
    final_df[fraud_rank_cols].max(axis=1)
)

final_df["fraud_weight_rank_range"] = (
    final_df["fraud_weight_worst_rank"]
    - final_df["fraud_weight_best_rank"]
)

fraud_weight_top100_stability = (
    final_df[
        final_df["fraud_10_rank"] <= 100
    ][
        [
            "merchant_abn",
            "merchant_name",
            "fraud_5_rank",
            "fraud_10_rank",
            "fraud_15_rank",
            "fraud_weight_rank_range",
        ]
    ]
)

fraud_weight_top100_stability[
    "fraud_weight_rank_range"
].describe()

count    100.000000
mean      14.820000
std       13.982225
min        0.000000
25%        5.000000
50%       10.000000
75%       22.000000
max       62.000000
Name: fraud_weight_rank_range, dtype: float64

In [37]:
fraud_weight_most_sensitive = (
    fraud_weight_top100_stability
    .sort_values(
        "fraud_weight_rank_range",
        ascending=False
    )
    .head(20)
)

fraud_weight_most_sensitive

,merchant_abn,merchant_name,fraud_5_rank,fraud_10_rank,fraud_15_rank,fraud_weight_rank_range
2165,57699602880,Tellus Id Institute,27,53,89,62
1621,45380641195,Nisl Arcu Iaculis Incorporated,52,77,113,61
1657,46012371285,A Ultricies Inc.,62,86,116,54
9,10323485998,Nunc Inc.,106,85,58,48
500,21025433654,Lorem Foundation,112,89,64,48
2417,63817709749,A Felis Ullamcorper Industries,64,84,103,39
56,11237511112,Magna Institute,60,80,99,39
1432,41271931352,Ac Sem Ut Company,79,96,117,38
3386,85362139954,In Consequat Associates,67,88,104,37
3450,86710922099,Ac Urna Consulting,81,58,45,36


### Fraud-Weight Sensitivity Interpretation

Top 100 overlap is 94 between 5% and 10% fraud weights, 86 between 5% and 15%, and 92 between 10% and 15%.

Among baseline Top 100 merchants, the rank range across these scenarios has median 10, mean 14.82 and maximum 62. Membership is relatively stable, while some exact ranks remain sensitive to risk preferences.

The 10% fraud weight is retained as a baseline decision-support assumption, not a statistically optimal weight.

### Final Weighting Framework

The final ranking combines business performance and fraud-risk safety.

The business score is based on five interpretable dimensions: merchant value, customer reach/scale, growth, stability, and market context. Fraud risk is then incorporated as a separate safety adjustment.

After flattening the two-stage weighting structure, the final effective weights are:

- **Merchant Value: 27%**
- **Customer Reach/Scale: 22.5%**
- **Growth: 18%**
- **Stability: 13.5%**
- **Market / Regional Context: 9%**
- **Fraud-Risk Safety: 10%**

The weighting scheme prioritises directly observed commercial performance while retaining supporting signals for future growth, consistency, socioeconomic context, and fraud risk.

Merchant value receives the highest weight because commercial value is the most direct indicator of BNPL onboarding potential. CustomerScore receives the second-highest weight and measures customer reach/scale through `unique_consumers`; repeat usage is not directly included. Growth captures future potential, while stability prevents highly volatile merchants from being rewarded solely for short-term growth.

The market component uses SEIFA IRSAD national decile with a lower weight because it describes the socioeconomic context of a merchant's customer base. Census and ATO variables are retained for diagnostics, not directly scored.

Fraud-risk safety receives a 10% baseline weight. This gives fraud risk a material influence on merchant selection without allowing it to dominate the ranking, recognising that KNN-estimated merchant risk is a relative signal with confidence and out-of-distribution limitations. Sensitivity analysis using 5%, 10%, and 15% fraud weights shows that the Top 100 remains reasonably stable under alternative risk preferences.

Highly correlated raw variables are consolidated into broader business dimensions before weighting to reduce double counting.

In [38]:
final_df.to_csv(
    PROJECT_ROOT
    / "member5_ranking"
    / "results"
    / "ranking_analysis_base.csv",
    index=False
)